# Your first model, built by hand

> A linear model in three lines of NumPy, a loss function that says what wrong means, and the discovery that finding good parameters is a search problem.

Read this chapter at `/learn/04-your-first-model/`. Exported from `src/content/chapters/04-your-first-model.mdx` — edit there, not here.


Today you write a model with no library doing anything on your behalf.

It's about twenty lines. It will not impress anybody at a party. And every neural
network in the rest of this book — including the ones with billions of
parameters — is a stack of exactly this thing. So it's worth doing slowly.

## A model is a function with adjustable numbers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n = 60
hours = rng.uniform(0, 10, n)                      # hours of revision
score = 12 + 7.5 * hours + rng.normal(0, 8, n)     # exam score, plus noise

plt.figure(figsize=(5, 3))
plt.scatter(hours, score, s=18)
plt.xlabel("hours revised"); plt.ylabel("exam score")
plt.tight_layout()

We made this data up, which means — unusually, and rather usefully — we know the
truth. It's `score = 12 + 7.5 * hours + noise`.

The model's job is to recover the `12` and the `7.5`, having seen only the dots.
It doesn't get to see the formula. Neither would you, in real life.

A linear model is one line of code. Here it is:

In [ ]:
def predict(x, w, b):
    """w: slope, b: intercept. Returns one prediction per element of x."""
    return w * x + b

predict(np.array([1.0, 5.0]), w=7.5, b=12.0)

That's a model. I know it doesn't look like much.

`w` and `b` are the **parameters** — the numbers we're hunting for. `x` is the
input, which is given to us and which we don't get to change.

That distinction is the entire subject, so let me put it plainly: **training
means holding the data fixed and moving the parameters.** Everything else is
detail.

`predict` is a pure function of `(x, w, b)`. What makes it a *model* rather than
just a function is that `w` and `b` are going to be **searched for** rather than
written down.

The signature you actually want is
`fn predict(x: &Array1<f64>, p: &Params) -> Array1<f64>`, where `Params` is the
thing an optimiser owns and mutates. Splitting the arguments that way — data
borrowed, parameters owned by the optimiser — is exactly the split every deep
learning framework makes.

## A loss says what "wrong" means

To guess parameters you need a way to compare guesses. That's a **loss
function**: one number, and lower is better.

In [ ]:
def mse(y_true, y_pred):
    return ((y_pred - y_true) ** 2).mean()

for w, b in [(7.5, 12.0), (5.0, 12.0), (7.5, 40.0), (0.0, 0.0)]:
    loss = mse(score, predict(hours, w, b))
    print(f"w={w:5.1f}  b={b:5.1f}   loss = {loss:9.2f}")

The true parameters get the lowest loss. Good — that's the property we needed,
and it's the only property we needed.

Mean squared error squares each miss and averages. The squaring
does two jobs: it removes the sign, so a miss of +5 and a miss of −5 can't cancel
each other out and pretend everything's fine, and it punishes large errors
disproportionately. Being wrong by 10 is *a hundred times* worse than being wrong
by 1, not ten times.

Choosing the loss is choosing what "wrong" means. That's a modelling decision
with real consequences, not a formality you inherit from a tutorial.

Squared error is a statement that outliers matter *enormously*. If your outliers
are genuine signal — actual fraud, actual failures — that's the right statement.
If they're measurement noise, you've just told your model to chase the noise, and
mean absolute error will serve you far better.

## Fitting, the stupid way

Before anything clever, let's do the obviously dumb thing: try lots of
parameters, keep the best one.

I'm not being facetious. This is a real technique for tiny parameter counts, and
more importantly, doing it once makes the next chapter *inevitable* rather than
magical.

In [ ]:
ws = np.linspace(0, 15, 120)
bs = np.linspace(-20, 40, 120)
W, B = np.meshgrid(ws, bs)

# Broadcasting does all the work: (120,120,1) against (60,) -> (120,120,60)
preds  = W[..., None] * hours + B[..., None]
losses = ((preds - score) ** 2).mean(axis=-1)     # collapse the examples axis

i, j = np.unravel_index(losses.argmin(), losses.shape)
print(f"best on grid:  w={ws[j]:.3f}  b={bs[i]:.3f}   loss={losses[i, j]:.2f}")
print(f"truth:         w=7.500  b=12.000")

Close! Not exact — the grid only has 120 rungs — but close.

And notice what turned a nested loop into one expression:
broadcasting against a new trailing axis, then
collapsing the example axis with `axis=-1`. Yesterday's
chapter, doing real work today.

Now — and this is the important bit — let's *look* at the thing we just searched.

In [ ]:
plt.figure(figsize=(5.2, 3.4))
plt.contourf(W, B, np.log(losses), levels=28, cmap="Blues_r")
plt.plot(ws[j], bs[i], "r*", markersize=13)
plt.xlabel("w (slope)"); plt.ylabel("b (intercept)")
plt.title("log loss surface"); plt.colorbar(label="log MSE")
plt.tight_layout()

There it is. That's the **loss landscape**, and it is the single most useful
mental image in all of machine learning. Please sit with it for a moment.

**Parameters are coordinates. Loss is altitude. Training is finding the bottom of
the valley.**

Every model you will ever train — a two-parameter line, a 400-billion-parameter
language model — is doing that. The valley is in a higher-dimensional space and
you can't draw it, but the picture stays honest. Keep this image; we'll be using
it constantly.

And here's why grid search dies, immediately and forever.

Two parameters at 120 values each is 14,400 evaluations. Instant. Ten parameters
is $120^{10}$ — about $6 \times 10^{20}$ evaluations, which is comfortably longer
than the age of the universe.

A *small* neural network has ten thousand parameters.

The curse of dimensionality isn't a subtlety or a footnote. It's the wall that
makes every brute-force approach useless, and it is the reason the rest of this
book has to exist.

## Fitting, the algebraic way

For this *particular* loss and this *particular* model family, it turns out
calculus just hands you the answer.

In [ ]:
X = np.column_stack([np.ones(n), hours])     # bias folded in as a column of 1s
theta = np.linalg.solve(X.T @ X, X.T @ score)
print(f"closed form:   b={theta[0]:.3f}  w={theta[1]:.3f}")
print(f"loss:          {mse(score, X @ theta):.3f}")

Two lines. Exact. No iteration, no learning rate, no waiting. Better than the
grid, instantly.

That column of ones is the trick from yesterday's exercise — it folds $b$ into
the same matrix multiply as $w$, so the bias needs no
special handling anywhere.

Write the model as $\hat{y} = X\theta$ and the loss as
$L(\theta) = \|X\theta - y\|^2$. At the minimum, the
gradient is zero:

$$
\nabla_\theta L = 2X^{\top}(X\theta - y) = 0
$$

Rearranging gives the **normal equations**:

$$
X^{\top}X\,\theta = X^{\top}y \qquad\Longrightarrow\qquad \theta = (X^{\top}X)^{-1}X^{\top}y
$$

Two notes for practice.

First: use `np.linalg.solve(A, b)` rather than `np.linalg.inv(A) @ b`. Solving is
faster *and* numerically far better behaved than forming an explicit inverse.
That's true in every language and it's worth making a lifelong habit.

Second: $X^{\top}X$ is $d \times d$ for $d$ features, and inverting it costs about
$O(d^3)$. At ten features that's free; at fifty thousand it's hopeless. And if
two of your features happen to be perfectly correlated, the matrix is singular
and there is no unique answer at all.

But the real limitation isn't cost — it's *scope*. This derivation works only
because the model is linear in $\theta$ and the loss is squared error. Put a
sigmoid on the output, add a hidden layer, switch to cross-entropy, and there is
no closed form. None. The algebra runs out.

That is precisely why gradient descent exists, and gradient descent works for all
of them. Tomorrow.

**"Why is the closed-form answer still not exactly 12 and 7.5?"** Because we added
noise, and the noise is genuinely in the data. The best possible line through
*these* dots is not the line that generated them. No method can fix that — it's
irreducible, and recognising it saves you weeks. More on this at the bottom of
the page.

**"`np.meshgrid` confused me."** It takes two 1-D arrays and gives back two 2-D
arrays holding every combination — so `W[i,j], B[i,j]` is one (w, b) pair. It's
how you turn "all pairs" into arrays you can do arithmetic on. Print `W.shape`
and `W[:2,:3]` and it'll click.

**"What does `W[..., None]` do?"** `None` in an index adds a new axis of size 1.
So `(120,120)` becomes `(120,120,1)`, which then broadcasts against `hours` of
shape `(60,)` to give `(120,120,60)` — every parameter pair evaluated on every
data point. It's the most compact way to say "all combinations."

**"I don't see why the loss surface is a bowl and not something lumpy."** For a
linear model with squared error it is *provably* a bowl — one minimum, no local
traps. That's a special property of this combination, and it's why the algebra
works. Neural networks are lumpy, and chapter 5 explains why we get away with it
anyway.

## From regression to classification

Same machinery, different output shape. Instead of predicting a number, predict a
*probability* — and for that, the unbounded score $wx + b$ has to be squashed
into $(0, 1)$ somehow.

In [ ]:
passed = (score > 50).astype(int)      # did they pass?

plt.figure(figsize=(5, 2.6))
plt.scatter(hours, passed, s=18, alpha=0.7)
plt.yticks([0, 1], ["fail", "pass"]); plt.xlabel("hours revised")
plt.tight_layout()

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
plt.figure(figsize=(5, 2.6))
plt.plot(z, sigmoid(z))
plt.axhline(0.5, ls=":", c="grey"); plt.axvline(0, ls=":", c="grey")
plt.title("sigmoid: any real number -> (0, 1)")
plt.tight_layout()

The sigmoid takes the raw score — which has a name, **logit** —
and returns something you're allowed to read as a probability. Feed it −1000 and
you get essentially 0; feed it +1000 and you get essentially 1; feed it 0 and you
get exactly a half.

A linear model with a sigmoid on the end is **logistic regression**. Which,
despite the name containing the word "regression," is a *classifier*. Sorry. It's
a historical accident and everyone has just decided to live with it.

It also remains the correct first thing to try on any binary problem, and it will
embarrass a surprising number of neural networks.

The loss has to change too, and for a rather good reason.

In [ ]:
def bce(y, p):                       # binary cross-entropy
    p = np.clip(p, 1e-12, 1 - 1e-12) # never take log(0)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"true label 1, predicted {p:.2f}   "
          f"squared error {(1-p)**2:6.3f}   cross-entropy {bce(np.array([1]), np.array([p])):7.3f}")

Look at the last row, and specifically at the two numbers on it.

The true answer is 1, and the model said 0.01. It is not merely wrong — it is
*confidently* wrong, which is much worse. Squared error shrugs and charges 0.98.
Cross-entropy charges 4.6, and that penalty grows
without bound as the predicted probability creeps toward zero.

That's the right incentive, and it's the whole reason cross-entropy is the loss
for classification. A model that says "definitely!" and is wrong should hurt far
more than one that said "maybe?" and was wrong.

Here's something I find genuinely satisfying. MSE and cross-entropy look like two
unrelated formulas somebody chose. They aren't. They're the same principle,
applied to two different assumptions.

The principle is maximum likelihood: *choose the
parameters that make your observed data most probable.*

**Assume the target is normally distributed around the prediction.** The
likelihood of one observation is
$p(y \mid x) \propto \exp\!\big(-(y - \hat{y})^2 / 2\sigma^2\big)$. Take the log,
drop the constants, flip the sign to turn maximisation into minimisation — and
what's left is $\sum (y - \hat{y})^2$. Squared error.

**Assume instead the target is a coin flip with probability $\hat{p}$.** The
likelihood of one observation is $\hat{p}^{\,y}(1-\hat{p})^{1-y}$. Take the log
and you get $y\log\hat{p} + (1-y)\log(1-\hat{p})$, which negated is *exactly* the
`bce` function in the cell above. Character for character.

Same principle. Different assumption about the noise. Two famous losses.

This is also why papers slide between "minimising the loss" and "maximising the
log-likelihood" mid-paragraph without explaining themselves. They're the same
activity with opposite signs, and everybody in the room already knows.

Now let's fit it, using the library for the moment:

In [ ]:
from sklearn.linear_model import LogisticRegression

X1 = hours.reshape(-1, 1)                 # sklearn wants (n_samples, n_features)
clf = LogisticRegression().fit(X1, passed)
w, b = clf.coef_[0, 0], clf.intercept_[0]

grid = np.linspace(0, 10, 200)
plt.figure(figsize=(5, 2.8))
plt.scatter(hours, passed, s=18, alpha=0.6)
plt.plot(grid, sigmoid(w * grid + b), c="crimson")
plt.axhline(0.5, ls=":", c="grey")
plt.xlabel("hours revised"); plt.ylabel("P(pass)")
plt.title(f"P(pass) = sigmoid({w:.2f}·hours {b:+.2f})")
plt.tight_layout()

Note the `reshape(-1, 1)`. scikit-learn always wants
`X` as `(n_samples, n_features)`, and handing it a bare 1-D array is the single
most common shape error in the entire library. Now you've met it once, on
purpose, in a safe place.

The model outputs a **probability**, not a class.

Turning 0.63 into "pass" requires a **threshold**, and 0.5 is a default, not a
law of nature. Where you put it trades false positives against false negatives,
and the right place depends entirely on what each one costs you.

Which means: a business decision has just been quietly handed to you, disguised
as a default argument. Notice when that happens.

In [ ]:
rng = np.random.default_rng(7)
m = 200
sqm       = rng.uniform(30, 150, m)
bedrooms  = rng.integers(1, 5, m).astype(float)
price     = 40 + 3.2 * sqm + 18 * bedrooms + rng.normal(0, 25, m)

# 1. Build X of shape (200, 3): a column of ones, then sqm, then bedrooms.
# 2. Solve for theta with the normal equation.
# 3. Print the recovered coefficients against the truth (40, 3.2, 18).
# 4. Compute the MSE, and compare it to a baseline that always predicts
#    price.mean(). By what factor is the model better?

print("replace me")

Everything you need is in the "normal equation" cell above — the only change is
that `np.column_stack` now takes three things instead of two.

For part 4, "by what factor" means divide one MSE by the other.

In [ ]:
X = np.column_stack([np.ones(m), sqm, bedrooms])
theta = np.linalg.solve(X.T @ X, X.T @ price)
print("recovered:", theta.round(2), "   truth: [40.  3.2 18. ]")

model_mse    = ((X @ theta - price) ** 2).mean()
baseline_mse = ((price.mean() - price) ** 2).mean()
print(f"model    MSE {model_mse:8.1f}")
print(f"baseline MSE {baseline_mse:8.1f}   ({baseline_mse / model_mse:.0f}x worse)")

Two things worth carrying away from this.

**The coefficients are close but not exact, and they can't be.** We added noise
with a standard deviation of 25, and no estimator on Earth can see through it.
That gap is the *irreducible error* term from
the bias–variance decomposition.

Having a rough sense of how big that term is for your problem is what stops you
spending a fortnight chasing an accuracy that was never on offer in the first
place. It's one of the most valuable and least taught intuitions in the field.

**The baseline comparison is the habit to keep.** "MSE 640" means nothing on its
own — it's not big or small, it's just a number in whatever units you happen to
be in.

"Forty times better than predicting the mean" is a *claim*. Every result in this
book, and every result in your work, should be quoted against a baseline. And
`always predict the mean` — or `always predict the majority class` — is the
baseline that costs you nothing to compute.

Tomorrow: what to do when there's no closed form — which is to say, what to do in
every single case that isn't this one.